# SafeAgent CPU baseline (local)

This notebook mirrors the CPU development path in `kaggle_baseline.ipynb`, but runs from your existing local checkout. It uses the repository's locked Python environment and configured data/model paths; it does not clone the repository, use Kaggle Secrets, or write to Kaggle-specific directories.

Select the project's Python 3.11 `.venv` as the notebook kernel, then run all cells. The data stages verify/recover the pinned public snapshot, prepare the fixed splits and validate them. The baseline trains on CPU and evaluates validation only; the test split remains reserved.

In [1]:
import json
from pathlib import Path
import subprocess
import sys

from taed2_safeagent.data.common import load_params


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "params.yaml").is_file()
            and (candidate / "taed2_safeagent" / "__init__.py").is_file()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find the SafeAgent project root above the notebook working directory"
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PARAMS_PATH = PROJECT_ROOT / "params.yaml"
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f"Select the project's Python 3.11 environment; current Python is {sys.version.split()[0]}"
    )

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

2026-10-16 13:57:40.041 | INFO     | taed2_safeagent.config:<module>:11 - PROJ_ROOT path is: /home/paugo/taed2-SafeAgent


Project root: /home/paugo/taed2-SafeAgent
Python: /home/paugo/taed2-SafeAgent/.venv/bin/python


## Choose the run mode

Keep `RUN_SHARED_EXPERIMENT` false for the local CPU run. This trains and evaluates without uploading a model or creating a shared MLflow run. Set it to true only from a clean, committed checkout after configuring the local DVC and MLflow credentials; that mode runs the formal tracked experiment instead of training twice.

In [2]:
RUN_SHARED_EXPERIMENT = False
if not isinstance(RUN_SHARED_EXPERIMENT, bool):
    raise TypeError("RUN_SHARED_EXPERIMENT must be true or false")

params = load_params(PARAMS_PATH)
data_paths = params["paths"]
print("Prepared data:", data_paths["processed"])
print("Reports:", data_paths["reports"])

Prepared data: /home/paugo/taed2-SafeAgent/data/processed/shell_safety
Reports: /home/paugo/taed2-SafeAgent/reports/data


## Recover and validate data

`fetch` downloads the exact pinned public source only if it is not already present, and verifies it if it is. The remaining commands reuse the project's audit, preparation and validation logic. No DVC remote credentials are required for this public-source path.

In [3]:
def run_module(module, *arguments):
    command = [sys.executable, "-m", module, *map(str, arguments)]
    print("$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)


for action in ("fetch", "audit", "prepare", "validate"):
    run_module("taed2_safeagent.dataset", action, "--params", PARAMS_PATH)

$ /home/paugo/taed2-SafeAgent/.venv/bin/python -m taed2_safeagent.dataset fetch --params /home/paugo/taed2-SafeAgent/params.yaml


2026-10-16 13:57:41.889 | INFO     | taed2_safeagent.config:<module>:11 - PROJ_ROOT path is: /home/paugo/taed2-SafeAgent
/home/paugo/taed2-SafeAgent/.venv/lib/python3.11/site-packages/deepchecks/core/serialization/dataframe/html.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/home/paugo/taed2-SafeAgent/.venv/lib/python3.11/site-packages/sklearn/metrics/_scorer.py:610: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


Raw snapshot checked
$ /home/paugo/taed2-SafeAgent/.venv/bin/python -m taed2_safeagent.dataset audit --params /home/paugo/taed2-SafeAgent/params.yaml


2026-10-16 13:58:19.286 | INFO     | taed2_safeagent.config:<module>:11 - PROJ_ROOT path is: /home/paugo/taed2-SafeAgent
/home/paugo/taed2-SafeAgent/.venv/lib/python3.11/site-packages/deepchecks/core/serialization/dataframe/html.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/home/paugo/taed2-SafeAgent/.venv/lib/python3.11/site-packages/sklearn/metrics/_scorer.py:610: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


Data audit complete
$ /home/paugo/taed2-SafeAgent/.venv/bin/python -m taed2_safeagent.dataset prepare --params /home/paugo/taed2-SafeAgent/params.yaml


2026-10-16 13:58:42.946 | INFO     | taed2_safeagent.config:<module>:11 - PROJ_ROOT path is: /home/paugo/taed2-SafeAgent
/home/paugo/taed2-SafeAgent/.venv/lib/python3.11/site-packages/deepchecks/core/serialization/dataframe/html.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/home/paugo/taed2-SafeAgent/.venv/lib/python3.11/site-packages/sklearn/metrics/_scorer.py:610: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


Data splits prepared
$ /home/paugo/taed2-SafeAgent/.venv/bin/python -m taed2_safeagent.dataset validate --params /home/paugo/taed2-SafeAgent/params.yaml


2026-10-16 13:59:02.966 | INFO     | taed2_safeagent.config:<module>:11 - PROJ_ROOT path is: /home/paugo/taed2-SafeAgent
/home/paugo/taed2-SafeAgent/.venv/lib/python3.11/site-packages/deepchecks/core/serialization/dataframe/html.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/home/paugo/taed2-SafeAgent/.venv/lib/python3.11/site-packages/sklearn/metrics/_scorer.py:610: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]
Data validation passed


## Train and evaluate the CPU baseline

The default path trains the fixed character TF-IDF and Linear SVM baseline, then evaluates only the validation split. If shared tracking is explicitly enabled, the experiment command performs the tracked run and upload as one operation instead.

In [4]:
if RUN_SHARED_EXPERIMENT:
    run_module(
        "taed2_safeagent.modeling.experiment", "--params", PARAMS_PATH
    )
else:
    run_module("taed2_safeagent.modeling.train", "--params", PARAMS_PATH)
    run_module("taed2_safeagent.modeling.evaluate", "--params", PARAMS_PATH)

metrics_path = PROJECT_ROOT / params["baseline"]["report_dir"] / "metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print("Validation rows:", metrics["rows"])
print("Validation macro F1:", metrics["macro_f1"])
print("DENY recall:", metrics["classes"]["DENY"]["recall"])
print("DENY predicted as ALLOW:", metrics["deny_to_allow_count"])
print("Metrics:", metrics_path)

$ /home/paugo/taed2-SafeAgent/.venv/bin/python -m taed2_safeagent.modeling.train --params /home/paugo/taed2-SafeAgent/params.yaml


2026-10-16 13:59:56.396 | INFO     | taed2_safeagent.config:<module>:11 - PROJ_ROOT path is: /home/paugo/taed2-SafeAgent


Baseline trained on 26660 examples
$ /home/paugo/taed2-SafeAgent/.venv/bin/python -m taed2_safeagent.modeling.evaluate --params /home/paugo/taed2-SafeAgent/params.yaml


2026-10-16 14:00:22.975 | INFO     | taed2_safeagent.config:<module>:11 - PROJ_ROOT path is: /home/paugo/taed2-SafeAgent


Validation macro F1: 0.861
DENY recall: 0.861
DENY predicted ALLOW: 19
Validation rows: 3333
Validation macro F1: 0.8612905322945594
DENY recall: 0.8612565445026178
DENY predicted as ALLOW: 19
Metrics: /home/paugo/taed2-SafeAgent/reports/baseline/metrics.json


## Results

The fitted model and reports are written to the paths configured in `params.yaml` under this checkout. The run does not evaluate the reserved test split. To repeat the run, use a fresh notebook kernel and run all cells again.